# Do not join: WDI GDP (current US$) vs OWID CO₂
Uses only the Python standard library.

The keys (`iso_code`, `year`) match. The quantities do not:
- GDP is **current US$** (market exchange rates), not PPP.
- CO₂ is **million tonnes**.
- Australia's WDI year is a **fiscal year** (ends 30 June), not a calendar year of emissions.

This notebook prints the mismatch and **refuses** a ratio. For per-capita CO₂ use the OWID population kit.


In [ ]:
import csv
from pathlib import Path

root = Path(".")
gdp_unit = None
gdp = {}
with (root / "data/samples/wb-gdp-current-sample.csv").open() as fh:
    for row in csv.DictReader(fh):
        gdp_unit = row["unit"]
        iso = (row["countryiso3code"] or "").upper()
        if len(iso) != 3 or iso == "WLD":
            continue
        gdp[(iso, int(row["date"]))] = float(row["value"])

co2 = {}
with (root / "data/samples/owid-co2-sample.csv").open() as fh:
    for row in csv.DictReader(fh):
        iso = (row["iso_code"] or "").upper()
        if len(iso) != 3 or iso.startswith("OWID_"):
            continue
        co2[(iso, int(row["year"]))] = float(row["co2"])

overlap = sorted(set(gdp) & set(co2))
print("gdp_unit", gdp_unit)
print("overlapping iso-year keys", overlap)
print("AUS 2020 GDP is a fiscal-year total; OWID CO2 is calendar year.")
print("REFUSE intensity = GDP / CO2. Current US$ is not PPP. FY is not calendar.")
assert gdp_unit.lower().startswith("current")
assert overlap, "samples should overlap on keys so the refusal is about units/vintage, not missing keys"
